In [17]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Part_4_aggregations").getOrCreate()
print("spark session ready")

spark session ready


In [2]:
patients_data = [
(1001,"Aarav Khan","Hyderabad",29,"Male","Gold","2023-01-10"),
(1002,"Priya Reddy","Bengaluru",34,"Female","Silver","2023-01-15"),
(1003,"Rahul Mehta","Mumbai",41,"Male","Gold","2023-02-02"),
(1004,"Sneha Kapoor","Delhi",26,"Female","Bronze","2023-02-18"),
(1005,"Kiran Patel","Ahmedabad",37,"Male","Silver","2023-03-01"),
(1006,"Ananya Das","Kolkata",31,"Female","Gold","2023-03-12"),
(1007,"Vikram Singh","Chennai",45,"Male","Bronze","2023-03-20"),
(1008,"Meera Nair","Kochi",28,"Female","Silver","2023-04-05"),
(1009,"Farhan Ali","Hyderabad",39,"Male","Gold","2023-04-15"),
(1010,"Divya Menon","Bengaluru",33,"Female","Silver","2023-04-21"),
(1011,"Arjun Iyer","Chennai",52,"Male","Gold","2023-05-01"),
(1012,"Neha Gupta","Delhi",25,"Female","Bronze","2023-05-11"),
(1013,"Sanjay Rao","Mumbai",48,"Male","Silver","2023-05-19"),
(1014,"Kavya Sharma","Hyderabad",30,"Female","Gold","2023-06-01"),
(1015,"Nikhil Verma","Pune",36,"Male","Silver","2023-06-12"),
(1016,"Ayesha Khan","Kolkata",27,"Female","Bronze","2023-06-20"),
(1017,"Manish Yadav","Lucknow",44,"Male","Gold","2023-07-05"),
(1018,"Pooja Shah","Ahmedabad",32,"Female","Silver","2023-07-18"),
(1019,"Rohan Nair","Kochi",40,"Male","Gold","2023-08-01"),
(1020,"Lakshmi Rao","Chennai",35,"Female","Silver","2023-08-14")
]
patients_columns = [
"patient_id","patient_name","city","age","gender","membership","registration_date"
]
patients_df = spark.createDataFrame(patients_data, patients_columns)

In [3]:
doctors_data = [
(201,"Dr Sameer Sharma","Cardiology","Hyderabad",1200),
(202,"Dr Kavita Iyer","Dermatology","Bengaluru",800),
(203,"Dr Imran Khan","Orthopedics","Mumbai",1500),
(204,"Dr Ramesh Reddy","Pediatrics","Delhi",900),
(205,"Dr Anita Mehta","Neurology","Hyderabad",2000),
(206,"Dr Joseph Mathew","Cardiology","Chennai",1300),
(207,"Dr Fatima Ali","Dermatology","Kolkata",850),
(208,"Dr Arvind Rao","Orthopedics","Bengaluru",1400),
(209,"Dr Leela Nair","Neurology","Kochi",1900),
(210,"Dr Ganesh Patil","General Medicine","Pune",700)
]
doctors_columns = [
"doctor_id","doctor_name","specialization","doctor_city","consultation_fee"
]
doctors_df = spark.createDataFrame(doctors_data, doctors_columns)

In [4]:
visits_data = [
(1,1001,201,"2024-03-01","Completed",2),
(2,1002,202,"2024-03-01","Completed",1),
(3,1003,203,"2024-03-02","Completed",3),
(4,1004,204,"2024-03-02","Pending",1),
(5,1005,206,"2024-03-03","Completed",2),
(6,1006,205,"2024-03-03","Completed",4),
(7,1007,207,"2024-03-04","Cancelled",1),
(8,1008,208,"2024-03-04","Completed",2),
(9,1009,201,"2024-03-05","Completed",1),
(10,1010,202,"2024-03-05","Completed",2),
(11,1011,205,"2024-03-06","Pending",3),
(12,1012,204,"2024-03-06","Completed",1),
(13,1013,203,"2024-03-07","Completed",2),
(14,1014,201,"2024-03-07","Completed",3),
(15,1015,210,"2024-03-08","Completed",1),
(16,1016,207,"2024-03-08","Cancelled",2),
(17,1017,209,"2024-03-09","Completed",4),
(18,1018,206,"2024-03-09","Completed",2),
(19,1019,209,"2024-03-10","Completed",3),
(20,1020,206,"2024-03-10","Pending",2),
(21,1001,205,"2024-03-11","Completed",3),
(22,1003,208,"2024-03-11","Completed",2),
(23,1006,201,"2024-03-12","Completed",1),
(24,1009,210,"2024-03-12","Completed",2),
(25,1014,202,"2024-03-13","Completed",1)
]
visits_columns = [
"visit_id","patient_id","doctor_id","visit_date","visit_status","tests_count"
]
visits_df = spark.createDataFrame(visits_data, visits_columns)

In [5]:
payments_data = [
(301,1,5200,"UPI","Paid"),
(302,2,2800,"Credit Card","Paid"),
(303,3,7500,"Cash","Paid"),
(304,4,2900,"UPI","Pending"),
(305,5,5300,"Debit Card","Paid"),
(306,6,10000,"Credit Card","Paid"),
(307,7,2850,"Cash","Cancelled"),
(308,8,5400,"UPI","Paid"),
(309,9,3200,"UPI","Paid"),
(310,10,4800,"Credit Card","Paid"),
(311,11,8000,"UPI","Pending"),
(312,12,2900,"Cash","Paid"),
(313,13,5500,"Credit Card","Paid"),
(314,14,7200,"UPI","Paid"),
(315,15,2700,"Debit Card","Paid"),
(316,16,4850,"Cash","Cancelled"),
(317,17,9900,"Credit Card","Paid"),
(318,18,5300,"UPI","Paid"),
(319,19,7900,"Debit Card","Paid"),
(320,20,5300,"UPI","Pending"),
(321,21,8000,"UPI","Paid"),
(322,22,5400,"Credit Card","Paid"),
(323,23,3200,"Cash","Paid"),
(324,24,4700,"UPI","Paid"),
(325,25,2800,"UPI","Paid")
]
payments_columns = [
"payment_id","visit_id","bill_amount","payment_mode","payment_status"
]
payments_df = spark.createDataFrame(payments_data, payments_columns)

In [ ]:
#Exercise 31
from pyspark.sql.functions import col,count ,avg, sum, max, min, to_date, udf,to_date
patients_df.groupBy("city") \
    .agg(count("*").alias("patient_count")) \
    .show()

In [ ]:
#Exercise 32
patients_df.groupBy("membership") \
    .agg(count("*").alias("patient_count")) \
    .show()

In [ ]:
#Exercise 33
doctors_df.groupBy("specialization") \
    .agg(count("*").alias("doctor_count")) \
    .show()

In [ ]:
#Exercise 34
visits_df.groupBy("visit_status") \
    .agg(count("*").alias("visit_count")) \
    .show()

In [ ]:
#Exercise 35
payments_df.groupBy("payment_mode") \
    .agg(count("*").alias("payment_count")) \
    .show()

In [ ]:
#Exercise 36
payments_df.agg(sum("bill_amount").alias("total_bill_amount")).show()

In [ ]:
#Exercise 37
payments_df.agg(avg("bill_amount").alias("avg_bill_amount")).show()

In [ ]:
#Exercise 38
final_df = visits_df.join(patients_df, "patient_id") \
                    .join(doctors_df, "doctor_id") \
                    .join(payments_df, "visit_id")
final_df.groupBy("city") \
    .agg(sum("bill_amount").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show()

In [ ]:
#Exercise 39
final_df.groupBy("specialization") \
    .agg(sum("bill_amount").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show()

In [ ]:
#Exercise 40
final_df.groupBy("doctor_name") \
    .agg(sum("bill_amount").alias("total_revenue")) \
    .orderBy(col("total_revenue").desc()) \
    .show()